In [1]:
pip install nltk conllu joblib

In [2]:
!git clone https://github.com/UniversalDependencies/UD_Haitian_Creole-Autogramm.git
!git clone https://github.com/UniversalDependencies/UD_Haitian_Creole-Adolphe.git

Cloning into 'UD_Haitian_Creole-Autogramm'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 249 (delta 187), reused 184 (delta 135), pack-reused 0 (from 0)
Receiving objects: 100% (249/249), 134.00 KiB | 4.96 MiB/s, done.
Resolving deltas: 100% (187/187), done.
Cloning into 'UD_Haitian_Creole-Adolphe'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 72 (delta 41), reused 46 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 2.00 MiB | 11.59 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [3]:
# @title Load and Parse Training Data (Combined Corpora)
import os
import glob
from conllu import parse

# Path configurations for both datasets
autogramm_split_folder = "/content/UD_Haitian_Creole-Autogramm/not-to-release/original_split"
adolphe_folder = "/content/UD_Haitian_Creole-Adolphe"

train_data = []

# --- 1. PARSE AUTOGRAMM TRAINING SPLITS ---
autogramm_files = glob.glob(os.path.join(autogramm_split_folder, "*.conllu"))
print(f"Found {len(autogramm_files)} Autogramm split files.")

for file_path in autogramm_files:
    with open(file_path, "r", encoding="utf-8") as f:
        sentences = parse(f.read())
        for sentence in sentences:
            sent_tuples = [
                (token["form"], token["upos"])
                for token in sentence
                if token["form"] and token["upos"]
            ]
            if sent_tuples:
                train_data.append(sent_tuples)

print(f"Loaded {len(train_data)} Autogramm sentences so far.")

# --- 2. PARSE ADOLPHE TRAINING & DEV FILES ---
# We include both train and dev files into training to maximize vocabulary depth
adolphe_files = [
    os.path.join(adolphe_folder, "ht_adolphe-ud-train.conllu"),
    os.path.join(adolphe_folder, "ht_adolphe-ud-dev.conllu"),
]

for file_path in adolphe_files:
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            sentences = parse(f.read())
            for sentence in sentences:
                sent_tuples = [
                    (token["form"], token["upos"])
                    for token in sentence
                    if token["form"] and token["upos"]
                ]
                if sent_tuples:
                    train_data.append(sent_tuples)
        print(f"Successfully added sentences from: {os.path.basename(file_path)}")
    else:
        print(f"Warning: File not found -> {file_path}")

print(f"\nTotal Combined Training Sentences: {len(train_data)}")

Found 18 Autogramm split files.
Loaded 144 Autogramm sentences so far.
Successfully added sentences from: ht_adolphe-ud-train.conllu
Successfully added sentences from: ht_adolphe-ud-dev.conllu

Total Combined Training Sentences: 3014


In [4]:
# @title Load and Parse Test Data
import os
from conllu import parse

test_file_path = "/content/UD_Haitian_Creole-Autogramm/ht_autogramm-ud-test.conllu"

test_data = []
if os.path.exists(test_file_path):
    with open(test_file_path, "r", encoding="utf-8") as f:
        test_sentences = parse(f.read())
        for sentence in test_sentences:
            sent_tuples = [
                (token["form"], token["upos"])
                for token in sentence
                if token["form"] and token["upos"]
            ]
            if sent_tuples:
                test_data.append(sent_tuples)
    print(f"Loaded {len(test_data)} testing sentences.")
else:
    print("Warning: Test file not found at the specified path.")

Loaded 144 testing sentences.


In [5]:
# @title Train the POS Tagger (Upgraded to Trigram)
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger

# 1. Base safety net (1-word window fallback)
t0 = DefaultTagger("NOUN")

# 2. Check individual words (1-word window)
t1 = UnigramTagger(train_data, backoff=t0)

# 3. Check word pairs (2-word window)
t2 = BigramTagger(train_data, backoff=t1)

# 4. Check three-word sequences (3-word window)
autogramm_tagger = TrigramTagger(train_data, backoff=t2)

print("Training complete with Trigram context layers!")

Training complete with Trigram context layers!


In [6]:
# @title Evaluate System Accuracy
if test_data:
    accuracy = autogramm_tagger.accuracy(test_data)
    print(f"\n🎯 Model Accuracy on Test Data: {accuracy * 100:.2f}%")


🎯 Model Accuracy on Test Data: 95.39%


In [7]:
# @title Save the Trained Model (using Joblib)
import os
import joblib
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger

model_filename = "haitian_creole_pos_tagger.joblib"

# --- SAVE THE TRAINED MODEL ---
# Joblib is ideal for saving NLTK tagger objects directly
with open(model_filename, "wb") as model_file:
    joblib.dump(autogramm_tagger, model_file)

file_size_kb = os.path.getsize(model_filename) / 1024
print(f"Model successfully saved as '{model_filename}' ({file_size_kb:.2f} KB)\n")

Model successfully saved as 'haitian_creole_pos_tagger.joblib' (75.74 KB)



In [8]:
# @title To test sentences using your newly trained model
import nltk
from nltk.tokenize import RegexpTokenizer

# A complex modern text about technology and society in Haiti
complex_text = (
    "Non pa'm se Speed "
    "map pote map la pou ou"
    "Teknoloji dijital la ap chanje jan moun kominike nan peyi Ayiti. "
    "Moun sèvi ak entènèt ak platfòm dijital tankou WhatsApp, Google, ak lòt "
    "rezo sosyal chak jou pou yo travay epi rete an kontak ak fanmi yo."
)
complex_text = "map la"

# We use a RegexpTokenizer to strip out punctuation cleanly without needing NLTK's 'punkt'
tokenizer = RegexpTokenizer(r"\w+|[^\w\s]")
tokens = tokenizer.tokenize(complex_text)

# Predict tags using your trained model
complex_predictions = autogramm_tagger.tag(tokens)

# Print the results cleanly line-by-line
print("--- Complex Text Predictions ---")
for word, tag in complex_predictions:
    print(f"{word:<15} -> {tag}")

--- Complex Text Predictions ---
map             -> NOUN
la              -> DET


=== Machine-readable metadata (DO NOT REMOVE!) ================================
Data available since: UD v2.13
License: CC BY-SA 4.0
Includes text: yes
Parallel: no
Genre: grammar-examples
Lemmas: manual native
UPOS: manual native
XPOS: not available
Features: manual native
Relations: converted from manual
Contributors: Pierre-Louis, Claudel; Jagodzińska, Sandra; Kahane, Sylvain; Savary, Agata; Schang, Emmanuel
Contributing: here
Contact: sylvain@kahane.fr
===============================================================================
=== Machine-readable metadata (DO NOT REMOVE!) ================================
Data available since: UD v2.16
License: CC BY-SA 4.0
Includes text: yes
Parallel: no
Genre: grammar-examples
Lemmas: converted with corrections
UPOS: converted with corrections
XPOS: not available
Features: converted with corrections
Relations: converted with corrections
Contributors: Adolphe, Jephtey
Contributing: here
Contact: ja983@scarletmail.rutgers.edu
===============================================================================